# Raw IB Low-Level Test

Minimales Notebook — direkte `ib_async`-Calls, **kein** `IBClient`-Wrapper.
Alle Parameter stehen in der ersten Code-Zelle.  
Jeder Block kann unabhängig ausgeführt werden (nach erfolgtem Connect).

## 0 — Setup & Parameter

In [17]:
# ── Verbindung ────────────────────────────────────────────────────────────────
HOST        = '127.0.0.1'
PORT        = 4001          # 4001 = Gateway Live, 4002 = Gateway Paper
                            # 7496 = TWS Live,     7497 = TWS Paper
CLIENT_ID   = 45            # andere ID als Haupt-Scanner verwenden!
MKT_DATA    = 1             # 1=live  2=frozen  3=delayed  4=delayed-frozen

# ── Stock ─────────────────────────────────────────────────────────────────────
SYMBOL      = 'SPY'
EXCHANGE    = 'ARCA'
PRIMARY_EX  = 'ARCA'        # für europäische Aktien wichtig
CURRENCY    = 'USD'

# ── Option (für Test 3) ───────────────────────────────────────────────────────
OPT_EXPIRY  = '20280121'    # YYYYMMDD
OPT_STRIKE  = 250.0
OPT_RIGHT   = 'P'           # 'P' = Put  |  'C' = Call
OPT_TRADING_CLASS = 'AAPL'   # aus Option-Chain-Test übernehmen (Test 2)

# ── Imports & Event-Loop ──────────────────────────────────────────────────────
from ib_async import IB, Stock, Option, util
import math, pprint

util.startLoop()
print('Setup OK')

Setup OK


## 1 — Connect

In [18]:
ib = IB()
ib.connect(HOST, PORT, clientId=CLIENT_ID, timeout=15)
ib.reqMarketDataType(MKT_DATA)

print(f'Connected : {ib.isConnected()}')
print(f'Server    : {ib.client.serverVersion()}')
print(f'MktData   : type {MKT_DATA}')

Connected : True
Server    : 178
MktData   : type 1


Peer closed connection.


---
## Test 1 — Stock-Preis

Drei unabhängige Methoden, damit sichtbar ist, welche davon Daten liefert.

| Methode | Wann verfügbar |
|---|---|
| `reqMktData` snapshot | US mit Delayed-Abo oder Live-Subscription |
| `reqHistoricalData` | Funktioniert für **alle** Börsen, kein Abo nötig |
| `reqMktData` stream | Gleiche Einschränkung wie Snapshot |

In [19]:
# ── Kontrakt qualifizieren ────────────────────────────────────────────────────
stk = Stock(SYMBOL, EXCHANGE, CURRENCY)

#tk=Stock('SPY','ARCA','USD')

stk.primaryExch = PRIMARY_EX
ib.qualifyContracts(stk)

print('Qualifizierter Kontrakt:')
print(f'  conId          : {stk.conId}')
print(f'  symbol         : {stk.symbol}')
print(f'  exchange       : {stk.exchange}')
print(f'  primaryExchange: {stk.primaryExchange}')
print(f'  currency       : {stk.currency}')
print(f'  localSymbol    : {stk.localSymbol}')
print(f'  tradingClass   : {stk.tradingClass}')

Qualifizierter Kontrakt:
  conId          : 756733
  symbol         : SPY
  exchange       : ARCA
  primaryExchange: ARCA
  currency       : USD
  localSymbol    : SPY
  tradingClass   : SPY


In [20]:
# ── Methode A: reqMktData snapshot ────────────────────────────────────────────
ticker_snap = ib.reqMktData(stk, '', snapshot=True, regulatorySnapshot=False)
ib.sleep(2.5)

print('=== reqMktData snapshot ===')
print(f'  last      : {ticker_snap.last}')
print(f'  bid       : {ticker_snap.bid}')
print(f'  ask       : {ticker_snap.ask}')
print(f'  close     : {ticker_snap.close}')
print(f'  marketPrice: {ticker_snap.marketPrice()}')

=== reqMktData snapshot ===
  last      : 708.48
  bid       : 708.47
  ask       : 708.51
  close     : 710.14
  marketPrice: 708.48


In [21]:
# ── Methode B: reqHistoricalData (kein Abo nötig) ─────────────────────────────
bars = ib.reqHistoricalData(
    stk,
    endDateTime='',        # letzter verfügbarer Zeitpunkt
    durationStr='5 D',     # 5 Handelstage zurück (überbrückt Feiertage)
    barSizeSetting='1 day',
    whatToShow='TRADES',
    useRTH=True,
    formatDate=1,
    keepUpToDate=False,
)

print('=== reqHistoricalData (Daily Bars) ===')
if bars:
    for b in bars:
        print(f'  {b.date}  O={b.open:.2f}  H={b.high:.2f}  L={b.low:.2f}  C={b.close:.2f}  Vol={b.volume}')
    print(f'\n→ Letzter Schlusskurs: {bars[-1].close:.2f} {CURRENCY}')
else:
    print('  Keine Bars erhalten.')

=== reqHistoricalData (Daily Bars) ===
  2026-04-14  O=687.69  H=694.58  L=687.66  C=694.46  Vol=8726061.0
  2026-04-15  O=695.26  H=700.27  L=694.29  C=699.94  Vol=6580173.0
  2026-04-16  O=701.13  H=702.77  L=698.53  C=701.66  Vol=6375659.0
  2026-04-17  O=706.15  H=712.37  L=705.76  C=710.14  Vol=7176921.0
  2026-04-20  O=708.79  H=709.90  L=706.14  C=708.72  Vol=5432878.0

→ Letzter Schlusskurs: 708.72 USD


In [22]:
# ── Methode C: reqMktData streaming (30 s) ────────────────────────────────────
ticker_stream = ib.reqMktData(stk, '', snapshot=False, regulatorySnapshot=False)
ib.sleep(4.0)

print('=== reqMktData streaming (nach 4 s) ===')
print(f'  last      : {ticker_stream.last}')
print(f'  bid       : {ticker_stream.bid}')
print(f'  ask       : {ticker_stream.ask}')
print(f'  close     : {ticker_stream.close}')
print(f'  marketPrice: {ticker_stream.marketPrice()}')

ib.cancelMktData(stk)
print('→ Stream abbestellt.')

=== reqMktData streaming (nach 4 s) ===
  last      : 708.51
  bid       : 708.5
  ask       : 708.53
  close     : 710.14
  marketPrice: 708.51
→ Stream abbestellt.


---
## Test 2 — Option-Chain-Parameter

`reqSecDefOptParams` liefert alle von IB gelisteten Expiries und Strikes für das Underlying.  
Kein Market-Data-Abo notwendig.

In [9]:
# ── rohe SecDefOptParams abrufen ──────────────────────────────────────────────
params = ib.reqSecDefOptParams(
    stk.symbol,
    '',             # futFopExchange (leer = Equity)
    stk.secType,    # 'STK'
    stk.conId,
)

print(f'{len(params)} Parametersets erhalten:')  
for p in params:
    print(f'  exchange={p.exchange:8s}  tradingClass={p.tradingClass:10s}  '
          f'expiries={len(p.expirations)}  strikes={len(p.strikes)}')

39 Parametersets erhalten:
  exchange=NASDAQOM  tradingClass=SPY         expiries=34  strikes=437
  exchange=AMEX      tradingClass=SPY         expiries=34  strikes=437
  exchange=CBOE2     tradingClass=2SPY        expiries=3  strikes=3
  exchange=NASDAQBX  tradingClass=SPY         expiries=34  strikes=437
  exchange=CBOE      tradingClass=2SPY        expiries=3  strikes=3
  exchange=MERCURY   tradingClass=2SPY        expiries=3  strikes=3
  exchange=CBOE      tradingClass=SPY         expiries=34  strikes=437
  exchange=MERCURY   tradingClass=SPY         expiries=34  strikes=437
  exchange=MEMX      tradingClass=2SPY        expiries=3  strikes=3
  exchange=BATS      tradingClass=2SPY        expiries=3  strikes=3
  exchange=PEARL     tradingClass=2SPY        expiries=3  strikes=3
  exchange=PHLX      tradingClass=2SPY        expiries=3  strikes=3
  exchange=EMERALD   tradingClass=2SPY        expiries=3  strikes=3
  exchange=ISE       tradingClass=2SPY        expiries=3  strikes=3
  exch

In [10]:
# ── SMART-Chain auswählen und inspizieren ─────────────────────────────────────
chain = next((p for p in params if p.exchange == 'SMART'), params[0] if params else None)

if chain is None:
    print('Keine Chain gefunden!')
else:
    expiries = sorted(chain.expirations)
    strikes  = sorted(chain.strikes)

    print(f'Exchange     : {chain.exchange}')
    print(f'TradingClass : {chain.tradingClass}')
    print(f'Expiries     : {len(expiries)} Stück')
    print(f'  erste 10   : {expiries[:10]}')
    print(f'  letzte 5   : {expiries[-5:]}')
    print(f'Strikes      : {len(strikes)} Stück')
    print(f'  Bereich    : {strikes[0]:.2f} – {strikes[-1]:.2f}')
    print(f'  Alle       : {strikes}')
    print()
    print(f'→ OPT_TRADING_CLASS für Test 3 setzen auf: "{chain.tradingClass}"')

Exchange     : SMART
TradingClass : 2SPY
Expiries     : 3 Stück
  erste 10   : ['20260605', '20260609', '20260612']
  letzte 5   : ['20260605', '20260609', '20260612']
Strikes      : 3 Stück
  Bereich    : 587.00 – 609.00
  Alle       : [587.0, 588.0, 609.0]

→ OPT_TRADING_CLASS für Test 3 setzen auf: "2SPY"


---
## Test 3 — Quote für eine einzelne Option

Direkte Qualifizierung + Market-Data-Request für genau einen Kontrakt.  
Strike und Expiry aus den Parametern oben wählen (Zelle 0).

In [11]:
# ── Option-Kontrakt bauen + qualifizieren ─────────────────────────────────────
opt = Option(
    symbol                       = SYMBOL,
    lastTradeDateOrContractMonth = OPT_EXPIRY,
    strike                       = OPT_STRIKE,
    right                        = OPT_RIGHT,
    exchange                     = EXCHANGE,
    currency                     = CURRENCY,
    tradingClass                 = OPT_TRADING_CLASS,
    multiplier                   = '100',
)

qualified = ib.qualifyContracts(opt)

print('=== Qualifizierter Options-Kontrakt ===')
if not opt.conId:
    print('  ❌ Qualifizierung fehlgeschlagen — Strike/Expiry existiert nicht bei IB.')
    print('     → Strike oder Expiry aus Test 2 anpassen.')
else:
    print(f'  conId          : {opt.conId}')
    print(f'  symbol         : {opt.symbol}')
    print(f'  expiry         : {opt.lastTradeDateOrContractMonth}')
    print(f'  strike         : {opt.strike}')
    print(f'  right          : {opt.right}')
    print(f'  exchange       : {opt.exchange}')
    print(f'  tradingClass   : {opt.tradingClass}')
    print(f'  multiplier     : {opt.multiplier}')
    print(f'  currency       : {opt.currency}')

=== Qualifizierter Options-Kontrakt ===
  conId          : 814191620
  symbol         : AAPL
  expiry         : 20280121
  strike         : 250.0
  right          : P
  exchange       : SMART
  tradingClass   : AAPL
  multiplier     : 100
  currency       : USD


In [22]:
# ── Market-Data-Snapshot (Bid / Ask / Last / Greeks) ─────────────────────────
# genericTickList '106' = implizite Vola + modelGreeks
if not opt.conId:
    print('Übersprungen — Kontrakt nicht qualifiziert.')
else:
    t = ib.reqMktData(opt, '106', snapshot=False, regulatorySnapshot=False)
    ib.sleep(3.0)

    print('=== Option Market Data ===')
    print(f'  bid        : {t.bid}')
    print(f'  ask        : {t.ask}')
    print(f'  last       : {t.last}')
    print(f'  close      : {t.close}')
    print(f'  volume     : {t.volume}')

    # Berechnetes Mid
    def _num(x):
        try: f = float(x); return f if not math.isnan(f) else None
        except: return None

    bid, ask = _num(t.bid), _num(t.ask)
    if bid and ask and ask >= bid:
        mid = (bid + ask) / 2
        spread_pct = (ask - bid) / mid * 100 if mid else None
        print(f'\n  mid        : {mid:.4f}')
        print(f'  spread     : {ask - bid:.4f}  ({spread_pct:.1f}%)')
    else:
        print('\n  mid        : n/a (kein Bid/Ask)')

    # Greeks
    greeks = t.modelGreeks or t.lastGreeks or t.bidGreeks or t.askGreeks
    print()
    print('=== Greeks ===')
    if greeks:
        print(f'  impliedVol   : {greeks.impliedVol}')
        print(f'  delta        : {greeks.delta}')
        print(f'  gamma        : {greeks.gamma}')
        print(f'  theta        : {greeks.theta}')
        print(f'  vega         : {greeks.vega}')
        print(f'  undPrice     : {greeks.undPrice}')
    else:
        print('  Keine Greeks erhalten (Delayed-Data liefert oft keine Greeks).')

    ib.cancelMktData(opt)
    print('\n→ Stream abbestellt.')

=== Option Market Data ===
  bid        : 24.35
  ask        : 25.1
  last       : 24.55
  close      : 25.48
  volume     : 6.0

  mid        : 24.7250
  spread     : 0.7500  (3.0%)

=== Greeks ===
  impliedVol   : 0.3018517353935777
  delta        : -0.2984374508683406
  gamma        : 0.003418441494807312
  theta        : -0.02101067872115081
  vega         : 1.2442506697710378
  undPrice     : 273.47601318359375

→ Stream abbestellt.


---
## Disconnect

In [14]:
ib.disconnect()
print('Disconnected:', not ib.isConnected())

Disconnected: True
